<a href="https://colab.research.google.com/github/zsroberts1/Southeast_FPV/blob/main/Southeast_FPV_PySAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install nrel-PySAM

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 MB 13.1 MB/s eta 0:00:00


In [ ]:
import json
import PySAM.Pvwattsv8 as pv
import pandas as pd
import csv


In [ ]:

kW_filename = open('/content/drive/MyDrive/DATA' ## csv folder containing the latitude and longitude of the location of interest (e.g., waterbodies)
                   '/ClimateCrisis_Coords.csv', 'r',encoding="utf-8-sig") ## file name
file = csv.DictReader(kW_filename)
Kw_list = []
Water_ID = []
weather_files = []
Annual_Energy_Output = []
CF = []
Global_Horiz_Irr = []
Solar_Capacity_kw = []
system_energy_yield = []
location_elevation = []
Lat = []
Long = []


In [ ]:
i = 0
for col in file:
    Kw_list.append(col['fpv_system_size_kw'])
    Water_ID.append(col['COMID'])
    weather_files.append(col['weather_files'])
    Lat.append(col['Lat'])
    Long.append(col['Lon'])

In [ ]:

## Creating a new instance of the Pvwatts 8 module to run
system_model = pv.new()

## gathering inputs from the pre-configured JSON file
with open('/content/drive/MyDrive/DATA/Southeast_pvwattsv8.json', 'r') as f: ## a JSON file pre-configured from SAM
    pv_inputs = json.load(f)

## iterate through the input key-value pairs and set the module inputs
for k, v in pv_inputs.items():
    if k != 'number_inputs':
      system_model.value(k, v)

for k in Kw_list:
    weather_path = ("/content/drive/MyDrive/DATA/Weather_data/New_weather_files" ## path to folder containing the weather files corresponding to each geographic location (e.g., waterbodies)
                    "/%s") % weather_files[i] ## weather file
    print(weather_path)
    print("these are the weather file: %s" % weather_files[i])
    system_model.SolarResource.solar_resource_file = weather_path

    capacity = float(Kw_list[i])  # kWdc
    system_model.SystemDesign.system_capacity = capacity
    print("this is the system capacity: %s" % Kw_list[i])
    i = i + 1
    print(i)

    system_model.SystemDesign.gcr = 0.7

    system_model.execute()

    annual_energy = system_model.Outputs.ac_annual
    capacity_factor = system_model.Outputs.capacity_factor_ac
    GHI = system_model.Outputs.gh
    system_size_kw = system_model.Outputs.gen
    energy_yield = system_model.Outputs.kwh_per_kw
    elevation = system_model.Outputs.elev


    Annual_Energy_Output.append(annual_energy)
    CF.append(capacity_factor)
    Global_Horiz_Irr.append(GHI)
    Solar_Capacity_kw.append(system_size_kw)
    system_energy_yield.append(energy_yield)
    location_elevation.append(elevation)



/content/drive/MyDrive/DATA/Weather_data/New_weather_files/36.291_-77.386.csv
these are the weather file: 36.291_-77.386.csv
this is the system capacity: 3078.000000000000000
1
/content/drive/MyDrive/DATA/Weather_data/New_weather_files/35.575_-75.387.csv
these are the weather file: 35.575_-75.387.csv
this is the system capacity: 28998.000000000000000
2
/content/drive/MyDrive/DATA/Weather_data/New_weather_files/35.583_-75.384.csv
these are the weather file: 35.583_-75.384.csv
this is the system capacity: 5050.000000000000000
3
/content/drive/MyDrive/DATA/Weather_data/New_weather_files/35.376_-75.283.csv
these are the weather file: 35.376_-75.283.csv
this is the system capacity: 3996.000000000000000
4
/content/drive/MyDrive/DATA/Weather_data/New_weather_files/35.332_-75.281.csv
these are the weather file: 35.332_-75.281.csv
this is the system capacity: 4886.000000000000000
5
/content/drive/MyDrive/DATA/Weather_data/New_weather_files/35.319_-75.284.csv
these are the weather file: 35.319_-

In [ ]:

dictionary = {'Water_ID': Water_ID, 'Lat': Lat, 'Lon': Long, 'Weather_Files': weather_files,
              'System_Size_kW': Kw_list, 'Year1_Energy_kWh': Annual_Energy_Output,
              'Capacity_Factor': CF, 'Energy_Yield': system_energy_yield, 'Elevation': location_elevation}
dataframe = pd.DataFrame(dictionary)
dataframe.to_csv('/content/drive/MyDrive/DATA/SAM_results'  ## output folder
                 '/Southeast_FPV_SAM_Results.csv ') ## output file